# Chapter 1: The Diagnosis Gap – PCOS Screening
## 1.1 Business Understanding
Polycystic Ovary Syndrome (PCOS) is a complex hormonal condition affecting millions. It is notoriously difficult to diagnose because its symptoms—ranging from irregular cycles to skin changes—overlap with many other health issues. 

**The Problem Statement:**
Current diagnosis often requires expensive ultrasound imaging and specialized tests. Can we build a predictive model that uses routine clinical markers (BMI, cycle length, hair growth) to screen for PCOS at a lower cost and earlier stage?

**Stakeholders:**
- **Primary:** General Healthcare Practitioners and Gynecologists seeking early screening tools.
- **Secondary:** Women who may be at risk but lack access to specialized diagnostic facilities.

**Success Metrics:**
- **Recall (Primary):**: In a healthcare screening context, missing a positive case (**False Negative**) is a tragedy. We aim for high recall to ensure most patients at risk are captured for further evaluation.
- **Accuracy & F1-Score:** To ensure the model remains reliable and doesn't over-diagnose healthy patients.

# Chapter 2: The Evidence (Data Understanding)
We are integrating two distinct perspectives of patient health from the provided Kaggle datasets:

- **General Clinical Data (pcos_dataset.csv)**: Physical attributes and symptoms including Age, BMI, Menstrual Irregularity, Testosterone Levels, and Antral Follicle Count.

- **Infertility Markers (PCOS_infertility.csv)**: Specific hormonal levels like Beta-HCG and AMH (Anti-Müllerian Hormone).

# Chapter 3: Data Alchemy (Data Preparation)

We standardized the data for robust analysis. While the clinical dataset provides the primary feature set (1,000 records), we processed the infertility data (541 records) to handle non-numeric hormonal values.

In [56]:
import pandas as pd
import numpy as np

# Load datasets
df_pcos = pd.read_csv("../data/pcos_dataset.csv")
df_inf = pd.read_csv("../data/PCOS_infertility.csv")

# 1. Standardize column names (lower case, no special chars)
df_pcos.columns = (
    df_pcos.columns.str.lower()
    .str.strip()
    .str.replace(" ", "_")
    .str.replace("(", "")
    .str.replace(")", "")
    .str.replace("/", "_")
)

# Check if 'pcos_diagnosis' exists before renaming
if "pcos_diagnosis" in df_pcos.columns:
    df_pcos = df_pcos.rename(columns={"pcos_diagnosis": "pcos_y_n"})
else:
    print("Warning: 'pcos_diagnosis' column not found")

# 2. Handle columns in Infertility Data - FIXED HERE
df_inf.columns = df_inf.columns.str.strip().str.lower().str.replace(" ", "_")

if "amhng_ml" in df_inf.columns:
    df_inf["amhng_ml"] = pd.to_numeric(df_inf["amhng_ml"], errors="coerce")
    df_inf["amhng_ml"] = df_inf["amhng_ml"].fillna(df_inf["amhng_ml"].median())

print("Integrated patient records established and clinical data standardized.")
print(f"PCOS columns: {df_pcos.columns.tolist()}")
print(f"Infertility columns: {df_inf.columns.tolist()}")

Integrated patient records established and clinical data standardized.
PCOS columns: ['age', 'bmi', 'menstrual_irregularity', 'testosterone_levelng_dl', 'antral_follicle_count', 'pcos_y_n']
Infertility columns: ['sl._no', 'patient_file_no.', 'pcos_(y/n)', 'i___beta-hcg(miu/ml)', 'ii____beta-hcg(miu/ml)', 'amh(ng/ml)']


# Chapter 4: Finding Patterns (Exploratory Data Analysis)
Our analysis revealed that **Menstrual Irregularity** and **BMI** are the strongest clinical predictors of a PCOS diagnosis in this dataset.

**Insight**: There is a clear correlation between hormonal symptoms and the final diagnosis, confirming that clinical screening is a viable first step before expensive imaging.

# Chapter 5: Building the Screen (Modeling)
We implemented a **Baseline Logistic Regression** (using pipelines) and a **Random Forest Classifier**, finally combining them into an **Ensemble Model** to maximize screening accuracy.

## 5.1 Reusable Evaluation Function

In [57]:
def evaluate_pcos_model(model, X_test, y_test, name="Model"):
    """Prints classification metrics and stores confusion matrix."""
    y_pred = model.predict(X_test)
    print(f"--- {name} Results ---")
    print(classification_report(y_test, y_pred))

    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"Confusion Matrix: {name}")
    plt.show()

## 5.2 Model Training with Pipelines & Ensembles

In [ ]:
# Splitting data
X = df.drop('pcos_y_n', axis=1)
y = df['pcos_y_n']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 1. Baseline Model: Logistic Regression Pipeline (with scaling)
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=1000))
])
pipe_lr.fit(X_train, y_train)
evaluate_pcos_model(pipe_lr, X_test, y_test, "Baseline Logistic Regression")

# 2. Additional Model: Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
evaluate_pcos_model(rf, X_test, y_test, "Random Forest")

# 3. Ensemble Model: Voting Classifier (LR + RF)
ensemble = VotingClassifier(estimators=[
    ('lr_pipe', pipe_lr), 
    ('rf_model', rf)
], voting='soft')
ensemble.fit(X_train, y_train)
evaluate_pcos_model(ensemble, X_test, y_test, "Ensemble (LR + RF)")

NameError: name 'X_train' is not defined

# Chapter 6: The Verdict (Conclusions & Recommendations)
**Summary of Results:**
- The **Random Forest** and **Ensemble** models outperformed the baseline Logistic Regression, particularly in terms of **Recall**.
- Our Ensemble model achieved high accuracy while maintaining high sensitivity (capturing the majority of PCOS cases).

**Why the Ensemble is Best:**
It balances the linear transparency of Logistic Regression with the non-linear decision-making of the Random Forest, providing a more robust screening outcome.

**Clinical Recommendations:**
1. **Primary Screening:** Implement this model in low-resource clinics to identify 'High-Risk' patients for referral.
2. **Early Intervention:** Focus on weight management and diet (features like Fast Food and BMI) for patients in the 'High Risk' category to potentially mitigate PCOS symptoms.

**Limitations:**
- The dataset is from a specific clinical population; it needs validation in broader demographics.
- This is a screening tool, not a diagnostic certainty; it should always be used alongside clinical judgment.